[Reference](https://medium.com/@umairali.khan/building-a-generic-knowledge-extraction-ai-framework-for-organization-specific-use-cases-cbb52ce93e48)

In [1]:
requirements = """
Extract project information from research grant proposals:
- Project title (string, required)
- Total budget in EUR (decimal, required)
- Start date (date, format: iso-date)
- Project status (enum: active, completed, pending)
- Principal investigator name (string)
"""

In [5]:
from __future__ import annotations

import json
from typing import List, Dict, Any, Optional
from pydantic import BaseModel

In [6]:
class FieldSpec(BaseModel):
    field_name: str                     # snake_case via validator
    field_type: AllowedTypes            # "str", "int", "decimal", "list[str]", etc.
    description: str
    required: bool
    enum: Optional[list[str]] = None
    pattern: Optional[str] = None
    format: Optional[Literal["iso-date", "currency-eur"]] = None

In [7]:
class ExtractionRequirements(BaseModel):
    use_case_name: str
    fields: list[FieldSpec]

In [8]:
class StructureAnalysis(BaseModel):
    structure_type: Literal["flat", "nested_list"]
    parent_container_name: str  # e.g., "line_items"
    parent_description: str
    item_description: str
    reasoning: str

In [9]:
prompt = """
Analyze the following extraction requirements and determine the output structure.

Use NESTED_LIST when:
- The DOCUMENT contains multiple items/records/rows to extract
- Instructions mention 'multiple items IN THE DOCUMENT', 'list of items', 'table of records'
- 'one line per item', 'one row per record', 'repeat for each entry'
- Document is structured as a table, list, or collection of similar items
- Example: Extract all products from an invoice (multiple products in one invoice)

Use FLAT when:
- ONE record per document (even if processing multiple documents)
- 'for each document', 'from each document', 'per document'
- Document describes a SINGLE entity (e.g., one project, one invoice, one person)
- Extracting summary/aggregate information from the document
- Example: Extract project details from grant document (one project per document)

IMPORTANT: 'For each X, extract...' means FLAT if X is the document itself,
NESTED if X refers to multiple items within the document.

Requirements:
{user_description}
"""

In [ ]:
from extractors import get_openai_config
from extractors.parsers import VisionParser

config = get_openai_config(use_azure=True)
parser = VisionParser(
    openai_config=config,
    use_context=True,      # Enable inter-page context
    dpi=300,               # Image resolution (200-300 recommended)
    clean_output=True      # Enable LLM-powered table merging
)

# Convert PDF to markdown
markdown_pages = parser.convert_pdf("document.pdf")
parser.save_markdown(markdown_pages, "output/document.md")